# v17: 物理候補生成 × CNN選択器

アイデア: **PFの逐次更新を「物理で作った候補軌跡の扇」+「学習による選択」に分解**する。

1. **生成(物理・GR非依存)**: 状態 `s=TVT+Z` の残差を区分線形dipでサンプリング → 候補TVTの扇
   (`cand_tvt = baseline_tvt + 滑らかな残差`)。baseline は既知dipの線形外挿。
2. **選択(学習)**: 各候補×各位置の **GR当てはまり**(observed GR vs typewell GR@候補TVT)を CNN に与え、
   位置ごとに候補を soft 選択 → `TVT_pred = Σ_p w[p,l]·cand_tvt[p,l]`。

**全データで2段確認**:
- (A) **オラクル診断**: 各坑井で「真TVTに最も近い1候補」を選んだときの overall RMSE = この候補族で到達可能な下限。
  → 物理の扇に正解が入っているか(改善余地があるか)を測る。
- (B) **CNN選択器の CV**: GroupKFold(by well), overall RMSE。

基準: RMSE≈15=無意味ライン(last_known複製), v9=LB6.946。 test互換(GR・幾何のみ, 真TVT非使用)。

> 実行はユーザー。P(候補数)/K(制御点)/step(残差幅)/crop は VRAM・速度で調整。

In [ ]:
import importlib, os, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
import nn_pf_common as C
importlib.reload(C)
print('device:', C.DEVICE)

wells = C.build_all('train')

In [ ]:
# ===== 物理候補生成 (GR非依存) =====
# 各坑井の評価ゾーンに対し, baseline_tvt(=s_base - Z) からの残差を区分線形dipでサンプル。
# 残差の制御点値 = 正規乱数の累積(=ランダムウォーク, 単調でない上下OK)。
# step を候補ごとに変えてマルチスケールに(扇のカバレッジを上げる)。

def eval_arrays(w):
    """評価ゾーンの静的配列(候補生成で再利用)。"""
    ev = np.where(w['evalm'])[0]
    if ev.size == 0:
        return None
    z = w['z'][ev].astype(np.float64)
    base_tvt = (w['s_base'][ev] - w['z'][ev]).astype(np.float64)
    md = w['md'][ev].astype(np.float64)
    span = max(md[-1] - md[0], 1.0)
    u = (md - md[0]) / span                         # [Lev] in [0,1]
    obs_gr = w['gr'][ev].astype(np.float64)
    mu, sd = obs_gr.mean(), obs_gr.std() + 1e-6
    true = w['tvt_true'][ev].astype(np.float64) if w['has_true'] else np.full(ev.size, np.nan)
    return dict(ev=ev, z=z, base_tvt=base_tvt, u=u, obs_gr=obs_gr, mu=mu, sd=sd,
                tw_tvt=w['tw_tvt'].astype(np.float64), tw_gr=w['tw_gr'].astype(np.float64), true=true)

def gen_candidates(ea, P=128, K=6, step_lo=2.0, step_hi=14.0, rng=None, include_baseline=True):
    """cand_tvt[P,Lev], feats[P,3,Lev] (grcost, residual/scale, slope), を返す。"""
    if rng is None:
        rng = np.random.default_rng(0)
    u = ea['u']; L = u.shape[0]
    frac = np.linspace(0.0, 1.0, K)
    steps = rng.uniform(step_lo, step_hi, size=(P, 1))                 # 候補ごとの残差幅
    inc = rng.standard_normal((P, K)) * steps                         # 制御点増分
    vals = np.cumsum(inc, axis=1); vals[:, 0] = 0.0                   # 評価開始で残差0(既知と連続)
    if include_baseline:
        vals[0, :] = 0.0                                             # 候補0 = baseline そのもの
    resid = np.stack([np.interp(u, frac, vals[p]) for p in range(P)])  # [P,L]
    cand_tvt = ea['base_tvt'][None, :] + resid                        # [P,L]
    # GR当てはまり: observed GR vs typewell GR@候補TVT (同じ坑井統計で標準化)
    tw_at = np.interp(cand_tvt.ravel(), ea['tw_tvt'], ea['tw_gr']).reshape(P, L)
    obs_z = (ea['obs_gr'][None, :] - ea['mu']) / ea['sd']
    tw_z = (tw_at - ea['mu']) / ea['sd']
    grcost = obs_z - tw_z                                             # [P,L]
    slope = np.gradient(cand_tvt, axis=1)                             # [P,L]
    feats = np.stack([grcost, resid / C.TARGET_SCALE, slope], axis=1)  # [P,3,L]
    return cand_tvt.astype(np.float32), feats.astype(np.float32)

In [ ]:
# ===== (A) オラクル診断: 物理の扇に正解は入っているか =====
# 各坑井で『真TVTに最も近い1候補』を選んだ場合の overall RMSE = この候補族の到達可能下限。
# (snippet の best_k と同じ発想を全データ・GR非依存で。)
def oracle_diag(wells, P=512, K=6, step_lo=2.0, step_hi=18.0, seed=0):
    errs = []
    rng = np.random.default_rng(seed)
    for w in wells:
        if not w['has_true']:
            continue
        ea = eval_arrays(w)
        if ea is None or not np.isfinite(ea['true']).any():
            continue
        cand, _ = gen_candidates(ea, P=P, K=K, step_lo=step_lo, step_hi=step_hi, rng=rng)
        t = ea['true']; m = np.isfinite(t)
        rmse_p = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, axis=1))  # [P]
        best = int(np.argmin(rmse_p))
        errs.append(cand[best, m] - t[m])
    e = np.concatenate(errs)
    print(f'oracle (best-1-candidate per well) overall RMSE = {np.sqrt(np.mean(e**2)):.4f}  (P={P})')
    print('  -> これより CNN選択器が良くなることはない(=改善余地の上限/下限の目安)。15を大きく下回れば筋が良い。')

oracle_diag(wells, P=512)

In [ ]:
# ===== oracle の内訳: per-well は小さいのに pooled(overall) が大きい理由を確認 =====
# overall RMSE = 全行プール(長い坑井ほど重い + 二乗で外れ坑井が支配)。
# 一部の「best候補でも大外しする坑井」が pooled を押し上げているはず。
def oracle_breakdown(wells, P=512, seed=0):
    rng = np.random.default_rng(seed)
    wid, ln, br, sse = [], [], [], []
    for w in wells:
        if not w['has_true']:
            continue
        ea = eval_arrays(w)
        if ea is None or not np.isfinite(ea['true']).any():
            continue
        cand, _ = gen_candidates(ea, P=P, rng=rng)
        t = ea['true']; m = np.isfinite(t)
        d = cand[:, m] - t[None, m]
        rmse_p = np.sqrt(np.mean(d ** 2, axis=1)); best = int(np.argmin(rmse_p))
        e = d[best]
        wid.append(w['wid']); ln.append(int(m.sum()))
        br.append(float(rmse_p[best])); sse.append(float(np.sum(e ** 2)))
    ln = np.array(ln); br = np.array(br); sse = np.array(sse)
    pooled = np.sqrt(sse.sum() / ln.sum())
    print(f'pooled overall oracle RMSE = {pooled:.3f}  (wells={len(br)})')
    print('per-well best_rmse 分位:', {f'p{q}': round(float(np.percentile(br, q)), 2) for q in [50, 75, 90, 95, 99]},
          '| max', round(float(br.max()), 1), '| mean', round(float(br.mean()), 2))
    order = np.argsort(br)[::-1]
    for frac in [0.05, 0.10, 0.20]:
        k = max(1, int(len(order) * frac))
        print(f'  最悪 {int(frac*100)}% 坑井({k}本) が pooled二乗誤差の {sse[order[:k]].sum()/sse.sum()*100:.0f}% を占める')
    print('  worst10 wid:', [wid[i] for i in order[:10]], '(plot_well(wid) で確認可)')
    return wid, ln, br, sse

_ = oracle_breakdown(wells, P=512)


In [ ]:
# ===== 可視化: 候補の扇・baseline・真値・ベスト候補 =====
import matplotlib.pyplot as plt

def baseline_overall(wells):
    """候補なしで baseline(dip線形外挿) だけの overall RMSE。oracle と比べる。"""
    errs = []
    for w in wells:
        if not w['has_true']:
            continue
        ea = eval_arrays(w)
        if ea is None:
            continue
        t = ea['true']; m = np.isfinite(t)
        if m.any():
            errs.append(ea['base_tvt'][m] - t[m])
    e = np.concatenate(errs)
    print(f'baseline-only overall RMSE = {np.sqrt(np.mean(e**2)):.4f}')
    print('  (これが oracle(15.43) とほぼ同じなら → 扇が無意味で baseline 自体がズレている=生成側の問題)')

baseline_overall(wells)

def plot_well(key, P=512, K=6, step_lo=2.0, step_hi=18.0, seed=0, n_show=300, ctx=400):
    w = wells[key] if isinstance(key, int) else next(x for x in wells if x['wid'] == key)
    ea = eval_arrays(w)
    cand, _ = gen_candidates(ea, P=P, K=K, step_lo=step_lo, step_hi=step_hi,
                             rng=np.random.default_rng(seed))
    t = ea['true']; m = np.isfinite(t)
    rmse_p = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, axis=1)); best = int(np.argmin(rmse_p))
    base_rmse = np.sqrt(np.mean((ea['base_tvt'][m] - t[m]) ** 2))
    x = ea['ev']                                   # 行インデックス
    known_idx = np.where(w['known'])[0]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    # 左: 扇全体
    ax[0].plot(x, cand[:n_show].T, color='red', alpha=0.04, lw=0.8)
    ax[0].plot(x, ea['base_tvt'], color='blue', lw=2, label='baseline (dip extrap)')
    ax[0].plot(x, t, color='black', lw=2, label='true TVT')
    if known_idx.size:
        kt = known_idx[known_idx >= known_idx[-1] - ctx]
        ax[0].plot(kt, w['tin'][kt], color='green', lw=1.5, label='known TVT')
    ax[0].set_title(f"{w['wid']}  candidate fan (P={P})")
    ax[0].set_xlabel('row idx'); ax[0].set_ylabel('TVT'); ax[0].legend()
    # 右: ベスト候補
    ax[1].plot(x, t, color='black', lw=2, label='true')
    ax[1].plot(x, ea['base_tvt'], color='blue', lw=1.5, label=f'baseline rmse={base_rmse:.2f}')
    ax[1].plot(x, cand[best], color='red', lw=2, alpha=0.8, label=f'best cand rmse={rmse_p[best]:.2f}')
    ax[1].set_title('best candidate vs true'); ax[1].legend()
    plt.tight_layout(); plt.show()
    print(f"{w['wid']}: eval_len={int(m.sum())} baseline_rmse={base_rmse:.2f} best_cand_rmse={rmse_p[best]:.2f}")

# 例: 最初の数坑井を表示（key を wid 文字列にすれば特定坑井も可）
for i in [0, 1, 2, 3]:
    plot_well(i)


In [ ]:
# ===== 改良ジェネレータ: 真の残差形状を PCA で学んだ低次元基底で候補を張る =====
# 狙い: oracle ~5。 random-walk(高次元・0中心)では真値を覆えないため、
#   残差 r = TVT_true - baseline_tvt の「形」を train から学び(mean+上位PC), その部分空間で候補生成。
#   ※ prior は true 残差を使うが test の真値は使わない(形のpriorを学ぶだけ=正当)。
#   ※ ただし下の診断は全wells in-sample(やや楽観)。CV で使うときは fold-train だけで prior を作ること(TODO)。

def build_residual_prior(wells, G=64, M=8):
    grid = np.linspace(0.0, 1.0, G)
    rows = []
    for w in wells:
        if not w['has_true']:
            continue
        ea = eval_arrays(w)
        if ea is None:
            continue
        t = ea['true']; m = np.isfinite(t)
        if m.sum() < 5:
            continue
        r = t - ea['base_tvt']
        rows.append(np.interp(grid, ea['u'][m], r[m]))   # 共通グリッドへ
    R = np.asarray(rows)
    mu = R.mean(0)
    X = R - mu
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    comps = Vt[:M]                                        # [M,G] 正規直交
    coeff_std = S[:M] / np.sqrt(len(R) - 1)               # 各PC係数の std(ft)
    var_ratio = float((S[:M] ** 2).sum() / (S ** 2).sum())
    print(f'prior: N={len(R)} wells, G={G}, M={M}, 累積寄与率={var_ratio:.3f}, coeff_std={np.round(coeff_std,1)}')
    return dict(grid=grid, mu=mu, comps=comps, coeff_std=coeff_std, M=M)

def gen_candidates_pca(ea, prior, P=128, rng=None, spread=4.0, include_mean=True):
    if rng is None:
        rng = np.random.default_rng(0)
    M, grid = prior['M'], prior['grid']
    a = rng.uniform(-spread, spread, size=(P, M)) * prior['coeff_std'][None, :]   # 係数を一様に張る
    resid_grid = prior['mu'][None, :] + a @ prior['comps']                        # [P,G]
    if include_mean:
        resid_grid[0] = prior['mu']
    u = ea['u']; L = u.shape[0]
    resid = np.stack([np.interp(u, grid, resid_grid[p]) for p in range(P)])       # [P,L]
    cand_tvt = ea['base_tvt'][None, :] + resid
    tw_at = np.interp(cand_tvt.ravel(), ea['tw_tvt'], ea['tw_gr']).reshape(P, L)
    obs_z = (ea['obs_gr'][None, :] - ea['mu']) / ea['sd']
    tw_z = (tw_at - ea['mu']) / ea['sd']
    grcost = obs_z - tw_z
    slope = np.gradient(cand_tvt, axis=1)
    feats = np.stack([grcost, resid / C.TARGET_SCALE, slope], axis=1)             # [P,3,L]
    return cand_tvt.astype(np.float32), feats.astype(np.float32)

PRIOR = build_residual_prior(wells, G=64, M=8)

# --- 投影 oracle: 真の残差を基底へ射影した最良(=この表現の床) ---
def projection_oracle(wells, prior):
    errs = []
    for w in wells:
        if not w['has_true']:
            continue
        ea = eval_arrays(w)
        if ea is None:
            continue
        t = ea['true']; m = np.isfinite(t)
        if m.sum() < 5:
            continue
        r = t - ea['base_tvt']
        rg = np.interp(prior['grid'], ea['u'][m], r[m])
        coef = (rg - prior['mu']) @ prior['comps'].T
        rec_grid = prior['mu'] + coef @ prior['comps']
        rec = np.interp(ea['u'], prior['grid'], rec_grid)
        errs.append((ea['base_tvt'] + rec)[m] - t[m])
    e = np.concatenate(errs)
    print(f'projection oracle (top-{prior["M"]}PC, 表現の床) overall RMSE = {np.sqrt(np.mean(e**2)):.4f}')

# --- サンプリング oracle: 実際に候補を生成して best-1 を選んだ場合(実用上限) ---
def sampling_oracle_pca(wells, prior, P=2000, spread=4.0, seed=0):
    rng = np.random.default_rng(seed); errs = []
    for w in wells:
        if not w['has_true']:
            continue
        ea = eval_arrays(w)
        if ea is None or not np.isfinite(ea['true']).any():
            continue
        cand, _ = gen_candidates_pca(ea, prior, P=P, rng=rng, spread=spread)
        t = ea['true']; m = np.isfinite(t)
        rmse_p = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, axis=1))
        errs.append(cand[int(np.argmin(rmse_p)), m] - t[m])
    e = np.concatenate(errs)
    print(f'sampling oracle (PCA, P={P}, spread={spread}) overall RMSE = {np.sqrt(np.mean(e**2)):.4f}')

projection_oracle(wells, PRIOR)
sampling_oracle_pca(wells, PRIOR, P=2000, spread=4.0)

# 以降の gen_candidates(CandDataset/oracle_diag/plot)を PCA版に差し替え(K/stepは無視)
def gen_candidates(ea, P=128, K=6, step_lo=2.0, step_hi=14.0, rng=None, include_baseline=True):
    return gen_candidates_pca(ea, PRIOR, P=P, rng=rng)


In [ ]:
# ===== Dataset / collate =====
CFG = dict(P=128, K=6, step_lo=2.0, step_hi=14.0, crop=1024)

class CandDataset(Dataset):
    def __init__(self, wells, mode='train', cfg=CFG, seed=0):
        self.cfg = cfg; self.mode = mode; self.seed = seed
        self.items = []
        for i, w in enumerate(wells):
            ea = eval_arrays(w)
            if ea is not None:
                self.items.append((w['wid'], ea))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        wid, ea = self.items[i]
        L = ea['u'].shape[0]
        # 学習はランダム seed(=毎エポック別候補=augment), 検証は固定 seed
        rng = np.random.default_rng(None if self.mode == 'train' else (self.seed * 100003 + i))
        cand, feats = gen_candidates(ea, P=self.cfg['P'], K=self.cfg['K'],
                                     step_lo=self.cfg['step_lo'], step_hi=self.cfg['step_hi'], rng=rng)
        true = ea['true'].astype(np.float32)
        # 学習時はクロップ
        if self.mode == 'train' and L > self.cfg['crop']:
            s = int(np.random.randint(0, L - self.cfg['crop'] + 1))
            sl = slice(s, s + self.cfg['crop'])
            cand = cand[:, sl]; feats = feats[:, :, sl]; true = true[sl]
        return dict(wid=wid,
                    cand=torch.from_numpy(cand),        # [P,l]
                    feats=torch.from_numpy(feats),      # [P,3,l]
                    true=torch.from_numpy(true))        # [l]

def cand_collate(batch):
    B = len(batch)
    P = batch[0]['cand'].shape[0]
    Cin = batch[0]['feats'].shape[1]
    L = max(b['cand'].shape[1] for b in batch)
    cand = torch.zeros(B, P, L)
    feats = torch.zeros(B, P, Cin, L)
    true = torch.full((B, L), float('nan'))
    valid = torch.zeros(B, L, dtype=torch.bool)
    for i, b in enumerate(batch):
        l = b['cand'].shape[1]
        cand[i, :, :l] = b['cand']
        feats[i, :, :, :l] = b['feats']
        true[i, :l] = b['true']
        valid[i, :l] = True
    return dict(cand=cand, feats=feats, true=true, valid=valid,
                wid=[b['wid'] for b in batch])

In [ ]:
# ===== CNN選択器 =====
# 各候補の系列特徴[Cin,L]を共有1D-CNN(dilated)で符号化→位置ごとのlogit→候補方向にsoftmax→重み付き候補TVT。
# 候補軸は置換不変(各候補を独立スコアリングしてsoftmaxで正規化)。
class SelectorCNN(nn.Module):
    def __init__(self, cin=3, h=48, dils=(1, 2, 4, 8, 16)):
        super().__init__()
        layers = [nn.Conv1d(cin, h, 5, padding=2), nn.GELU()]
        for d in dils:
            layers += [nn.Conv1d(h, h, 3, padding=d, dilation=d), nn.GELU()]
        self.cnn = nn.Sequential(*layers)
        self.head = nn.Conv1d(h, 1, 1)
        self.log_temp = nn.Parameter(torch.tensor(0.0))

    def forward(self, batch):
        cand = batch['cand']                      # [B,P,L]
        feats = batch['feats']                    # [B,P,Cin,L]
        B, P, Cin, L = feats.shape
        x = feats.reshape(B * P, Cin, L)
        z = self.head(self.cnn(x)).reshape(B, P, L)   # [B,P,L] logit
        w = torch.softmax(z * F.softplus(self.log_temp), dim=1)
        pred = (w * cand).sum(1)                   # [B,L]
        return pred, w

# スモークテスト
ds = CandDataset(wells[:8], mode='train')
dl = DataLoader(ds, batch_size=4, collate_fn=cand_collate)
b = {k: (v.to(C.DEVICE) if torch.is_tensor(v) else v) for k, v in next(iter(dl)).items()}
mdl = SelectorCNN().to(C.DEVICE)
pred, w = mdl(b)
print('pred:', tuple(pred.shape), '| weights:', tuple(w.shape))

In [ ]:
# ===== (B) CV 学習 (GroupKFold by well, overall RMSE) =====
def huber_eval_loss(pred, batch, delta=10.0):
    m = batch['valid'] & torch.isfinite(batch['true'])
    if m.sum() == 0:
        return pred.sum() * 0.0
    err = (pred - torch.nan_to_num(batch['true']))[m]
    return F.huber_loss(err, torch.zeros_like(err), delta=delta)

def to_dev(b):
    return {k: (v.to(C.DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}

@torch.no_grad()
def eval_oof(model, dl):
    model.eval(); preds, trues = [], []
    for b in dl:
        b = to_dev(b); pred, _ = model(b)
        m = (b['valid'] & torch.isfinite(b['true']))
        preds.append(pred[m].cpu().numpy()); trues.append(b['true'][m].cpu().numpy())
    return preds, trues

def run_cv_v17(wells, cfg=CFG, n_splits=5, epochs=20, lr=1e-3, batch_size=8, seed=42):
    groups = np.array([w['wid'] for w in wells])
    gkf = GroupKFold(n_splits=n_splits)
    oof = {}; fold_best = []
    for fold, (tr, va) in enumerate(gkf.split(np.arange(len(wells)), groups=groups)):
        tr_ds = CandDataset([wells[i] for i in tr], mode='train', cfg=cfg, seed=seed + fold)
        va_ds = CandDataset([wells[i] for i in va], mode='eval', cfg=cfg, seed=seed + fold)
        tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=cand_collate)
        va_dl = DataLoader(va_ds, batch_size=1, shuffle=False, collate_fn=cand_collate)
        torch.manual_seed(seed + fold)
        model = SelectorCNN().to(C.DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        best, best_state = float('inf'), None
        for ep in range(epochs):
            model.train(); tot = 0.0; nb = 0
            for b in tr_dl:
                b = to_dev(b); pred, _ = model(b); loss = huber_eval_loss(pred, b)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
                tot += float(loss); nb += 1
            sch.step()
            preds, trues = eval_oof(model, va_dl)
            rmse = C.overall_rmse(preds, trues)
            if rmse < best:
                best = rmse; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f'fold{fold} ep{ep:02d} loss={tot/max(nb,1):.4f} val_overall_rmse={rmse:.4f}')
        model.load_state_dict(best_state)
        # OOF 収集
        model.eval()
        with torch.no_grad():
            for b in va_dl:
                wid = b['wid'][0]; bb = to_dev(b); pred, _ = model(bb)
                m = (bb['valid'] & torch.isfinite(bb['true']))[0]
                oof[wid] = (pred[0][m].cpu().numpy(), bb['true'][0][m].cpu().numpy())
        fold_best.append(best)
        print(f'fold{fold} best val_overall_rmse={best:.4f}')
    preds = [oof[w['wid']][0] for w in wells if w['wid'] in oof]
    trues = [oof[w['wid']][1] for w in wells if w['wid'] in oof]
    cv = C.overall_rmse(preds, trues)
    print(f'=== v17 CV overall RMSE = {cv:.4f} (fold best mean {np.mean(fold_best):.4f}) ===')
    return cv, oof

cv, oof = run_cv_v17(wells, epochs=20, batch_size=8)
print('CV overall RMSE (v17):', cv)

In [ ]:
os.makedirs('artifacts_v17', exist_ok=True)
with open('artifacts_v17/oof_v17.pkl', 'wb') as f:
    pickle.dump({'cv': cv, 'oof': oof}, f)
print('saved artifacts_v17/oof_v17.pkl')